In [4]:
import pandas as pd

In [5]:
data = pd.read_csv('./../tmpsnx71rnm.csv')
data.columns

Index(['Unnamed: 0', 'Rank (Borda)', 'Model', 'Zero-shot',
       'Active Parameters (B)', 'Total Parameters (B)', 'Embedding Dimensions',
       'Max Tokens', 'Mean (Task)', 'Mean (TaskType)', 'Bitext Mining',
       'Classification', 'Clustering', 'Instruction Reranking',
       'Multilabel Classification', 'Pair Classification', 'Reranking',
       'Retrieval', 'STS'],
      dtype='str')

In [6]:
data.drop(columns=['Unnamed: 0', 'Zero-shot','Active Parameters (B)','Classification', 'Clustering', 'Instruction Reranking',
       'Multilabel Classification', 'Pair Classification', 'STS', 'Mean (Task)', 'Mean (TaskType)', 'Bitext Mining'], inplace=True)

In [7]:
data['Model'] = data['Model'].apply(lambda x: x.split('(')[0].strip().replace('[', '').replace(']', ''))

In [8]:
data.isnull().sum()

Rank (Borda)              0
Model                     0
Total Parameters (B)     48
Embedding Dimensions     16
Max Tokens               35
Reranking               237
Retrieval               234
dtype: int64

In [9]:
for col in data.columns:
    data[col] = data[col].fillna('0.00')
    if data[col].dtype == 'object':
        data[col]=data[col].astype(float)


In [10]:
filtered = data[
    (data['Total Parameters (B)'] != 0.00) &
    (data['Total Parameters (B)'] < 0.25)# & (data['Total Parameters (B)'] < 2.0)
]
sorted_data = filtered.sort_values('Total Parameters (B)', ascending=True)


In [11]:
sorted_data= sorted_data.sort_values('Retrieval', ascending=False)

In [12]:
sorted_data.head(10)

,Rank (Borda),Model,Total Parameters (B),Embedding Dimensions,Max Tokens,Reranking,Retrieval
19,19,jina-embeddings-v5-text-nano,0.212,768.0,8192.0,64.63,63.26
82,83,granite-embedding-97m-multilingual-r2,0.097,384.0,8192.0,59.39,60.32
57,58,F2LLM-v2-160M,0.159,640.0,40960.0,60.34,54.08
65,66,multilingual-e5-small,0.118,384.0,512.0,60.43,50.91
68,69,F2LLM-v2-80M,0.080,320.0,40960.0,58.95,50.13
59,60,bilingual-embedding-small,0.118,384.0,512.0,59.31,49.55
83,84,granite-embedding-107m-multilingual,0.107,384.0,512.0,58.48,48.08
113,114,static-similarity-mrl-multilingual-v1,0.108,1024.0,0.0,49.45,41.21
87,88,nomic-embed-text-v1-ablated,0.137,768.0,8192.0,45.04,40.74
98,99,nomic-embed-text-v1-unsupervised,0.137,768.0,8192.0,48.18,40.66


In [13]:
sorted_data.to_csv('./../filtered_sorted_models.csv', index=False)

In [14]:
import os
os.chdir('./..')

In [15]:
from src.utils import log, CustomException
log = log()

1. Based on research I chose bge-base-en-v1.5.
Features:
    - 0.1B parameters 
    - model size is 430MB approx.
    - 512 MAX tokens
    - Retriever effieciency is also good.

In [16]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('BAAI/bge-base-en-v1.5')
tokenizer = model.tokenizer

/home/vraj/.conda/envs/eu-mdr/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 14827.44it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [17]:
text = "What is the capital of France? The capital of France is Paris. You can also visit the Eiffel Tower in Paris."
token_count = tokenizer.encode(text)
print(f"Number of tokens: {len(token_count)}")

Number of tokens: 28


In [21]:
import sys
import json
try:
    log.info("Loading cleaned EU MDR 2017-745 documents for token length analysis.")
    with open('data/processed/cleaned_eu_mdr_2017-745.json', 'r') as f:
        docs = json.load(f)
    
    structure_prefix = ""
    part_pattern = "\nPART [A-Z] \n"
    simple_pattern = r'\n(\d+)\.\s*\n'
    decimal_pattern = "\n\d+\.\d+\.\s*\n"
    triple = "\n\d+\.\d+\.\d+\.\s*\n"

    for i, doc in enumerate(docs):
        page_content = doc.get('page_content')
        metadata = doc.get('metadata')
        token_length = len(tokenizer.encode(page_content))
      

            
except Exception as e:
    log.exception(f"An error occurred: {e}")
    raise CustomException(e, sys)

2026-06-03 11:17:41,282 53 3204945777 - INFO - Loading cleaned EU MDR 2017-745 documents for token length analysis.
Token indices sequence length is longer than the specified maximum sequence length for this model (11358 > 512). Running this sequence through the model will result in indexing errors


In [22]:
for doc in docs:
    doc.get('page_content', '')

In [119]:
info = docs[148].get('page_content', '')

In [120]:
import re
pattern = r'\n(\d+)\.\s*\n(.+)'
part_pattern = "\nPART [A-Z] \n(.+)"
matches =  re.finditer(part_pattern, docs[147].get('page_content'))

In [121]:
len(tokenizer.encode(docs[147].get('page_content', '')))

4807

In [319]:
patterns = {"part" : "\nPART [A-Z] \n(.+)", "simple" : r'\n(\d+)\.\s*\n(.+)', "decimal" : "\n\d+\.\d+\.\s*\n(.+)",
"triple" : r'^(\n\d+(?:\.\d+)*\.?)\s{2,}(.+)$'}
marker_patterns = {"pattern_annex" : "^(ANNEX [IVX]+) \n(.+)", "pattern_chapter" : "^(CHAPTER [IVX]+) \n(.+)", "pattern_section" : "^(SECTION [0-9]+) \n(.+)", "pattern_article" :"^(Article [0-9]+) \n(?!Article)(?!— )(.+)"}
def token_length(page_content):
    return len(tokenizer.encode(page_content))
def get_split_levels(page_content):
    levels = []
    if token_length(page_content) <= 512:
        return ['no_split']
    if re.search(patterns['part'], page_content):
        levels.append("part")
    if re.search(patterns['simple'], page_content):
        levels.append("simple")
    if re.search(patterns['decimal'], page_content):
        levels.append("decimal")
    if re.search(patterns['triple'], page_content):
        levels.append("triple")
    return levels

In [ ]:
def get_text_piece(pattern, text):
    find = re.finditer(pattern, text, re.M)
    matches = []
    pieces = []
    for match in find:
        matches.append((match.start(), match.group()))
    if not matches:
        return [text]
    if matches and matches[0][0] > 0:
        pieces.insert(0, text[0:matches[0][0]])
    for i, mark in enumerate(matches):
        pattern_length = len(mark[1])
        if i < len(matches)-1:
            pieces.append(text[mark[0]:matches[i+1][0]])
        else:
            pieces.append(text[mark[0]:])
        
    return pieces

a = get_text_piece(patterns['part'], info)
lengths = [token_length(piece) for piece in a]
a


['ANNEX VI \nINFORMATION TO BE SUBMITTED UPON THE REGISTRATION OF DEVICES AND ECONOMIC \nOPERATORS IN ACCORDANCE WITH ARTICLES 29(4) AND 31, CORE DATA ELEMENTS TO BE PROVIDED \nTO THE UDI DATABASE TOGETHER WITH THE UDI-DI IN ACCORDANCE WITH ARTICLES 28 AND 29, \nAND THE UDI SYSTEM ',
 '\nPART A \nINFORMATION TO BE SUBMITTED UPON THE REGISTRATION OF DEVICES AND ECONOMIC \nOPERATORS IN ACCORDANCE WITH ARTICLES 29(4) AND 31 \nManufacturers or, when applicable, authorised representatives, and, when applicable, importers shall submit the \ninformation referred to in Section 1 and shall ensure that the information on their devices referred to in Section 2 is \ncomplete, correct and updated by the relevant party. \n1.  \nInformation relating to the economic operator \n1.1.  \ntype of economic operator(manufacturer, authorised representative, or importer), \n1.2.  \nname, address and contact details of the economic operator, \n1.3.  \nwhere submission of information is carried out by another p

In [321]:
if re.match(patterns['part'], a[0]).group():
    print("ok")

AttributeError: 'NoneType' object has no attribute 'group'

In [322]:
any(re.match(p, x) for p in marker_patterns.values() for x in a )

True

In [323]:
S = []
for s in a:
    if token_length(s) > 512:
        S.append(get_text_piece(patterns['simple'], s))
S

[['\nPART A \nINFORMATION TO BE SUBMITTED UPON THE REGISTRATION OF DEVICES AND ECONOMIC \nOPERATORS IN ACCORDANCE WITH ARTICLES 29(4) AND 31 \nManufacturers or, when applicable, authorised representatives, and, when applicable, importers shall submit the \ninformation referred to in Section 1 and shall ensure that the information on their devices referred to in Section 2 is \ncomplete, correct and updated by the relevant party. ',
  '\n1.  \nInformation relating to the economic operator \n1.1.  \ntype of economic operator(manufacturer, authorised representative, or importer), \n1.2.  \nname, address and contact details of the economic operator, \n1.3.  \nwhere submission of information is carried out by another person on behalf of any of the economic operators \nmentioned under Section 1.1, the name, address and contact details of that person, \n1.4.  \nname address and contact details of the person or persons responsible for regulatory compliance referred to in \nArticle 15. ',
  '\n2

In [324]:
v = any(re.match(p,g) for p in patterns.values() for k in S for g in k)
v

True

In [325]:
D = []
for d in S:
    for piece in d:
        print(f"Processing piece with token length: {token_length(piece)}")
        if token_length(piece) > 512:
            D.append(get_text_piece(patterns['decimal'], piece))
D

Processing piece with token length: 76
Processing piece with token length: 108
Processing piece with token length: 455
Processing piece with token length: 8
Processing piece with token length: 707
Processing piece with token length: 105
Processing piece with token length: 516
Processing piece with token length: 757
Processing piece with token length: 337
Processing piece with token length: 1221


[["\n1.  \nDefinitions \nAutomatic identification and data capture ('AIDC') \nAIDC is a technology used to automatically capture data. AIDC technologies include bar codes, smart cards, \nbiometrics and RFID. \nBasic UDI-DI \nThe Basic UDI-DI is the primary identifier of a device model. It is the DI assigned at the level of the device unit \nof use. It is the main key for records in the UDI database and is referenced in relevant certificates and EU \ndeclarations of conformity. \nUnit of Use DI \nThe Unit of Use DI serves to associate the use of a device with a patient in instances in which a UDI is not \nlabelled on the individual device at the level of its unit of use, for example in the event of several units of the \nsame device being packaged together. \nConfigurable device \nA configurable device is a device that consists of several components which can be assembled by the \nmanufacturer in multiple configurations. Those individual components may be devices in themselves. \nConfig

In [326]:
T=[]
for t in D:
    for piece in t:
        #print(f"Processing piece with token length: {token_length(piece)}")
        if token_length(piece) > 512:
            T.append(get_text_piece(patterns['triple'], piece))
T

[["\n1.  \nDefinitions \nAutomatic identification and data capture ('AIDC') \nAIDC is a technology used to automatically capture data. AIDC technologies include bar codes, smart cards, \nbiometrics and RFID. \nBasic UDI-DI \nThe Basic UDI-DI is the primary identifier of a device model. It is the DI assigned at the level of the device unit \nof use. It is the main key for records in the UDI database and is referenced in relevant certificates and EU \ndeclarations of conformity. \nUnit of Use DI \nThe Unit of Use DI serves to associate the use of a device with a patient in instances in which a UDI is not \nlabelled on the individual device at the level of its unit of use, for example in the event of several units of the \nsame device being packaged together. \nConfigurable device \nA configurable device is a device that consists of several components which can be assembled by the \nmanufacturer in multiple configurations. Those individual components may be devices in themselves. \nConfig

In [ ]:
f = T[0]
l = get_split_levels(f[0])
print(l)
if token_length(f[0])<=512:
    chu = [f[0]]
for q in f[0]:
    

['simple']


In [282]:
w = ["\n1.  \nDefinitions \nAutomatic identification and data capture ('AIDC') \nAIDC is a technology used to automatically capture data. AIDC technologies include bar codes, smart cards, \nbiometrics and RFID. \nBasic UDI-DI \nThe Basic UDI-DI is the primary identifier of a device model. It is the DI assigned at the level of the device unit \nof use. It is the main key for records in the UDI database and is referenced in relevant certificates and EU \ndeclarations of conformity. \nUnit of Use DI \nThe Unit of Use DI serves to associate the use of a device with a patient in instances in which a UDI is not \nlabelled on the individual device at the level of its unit of use, for example in the event of several units of the \nsame device being packaged together. \nConfigurable device \nA configurable device is a device that consists of several components which can be assembled by the \nmanufacturer in multiple configurations. Those individual components may be devices in themselves. \nConfigurable devices include computed tomography (CT) systems, ultrasound systems, anaesthesia systems, \nphysiological Monitoring systems, radiology information systems (RIS). \nConfiguration \nConfiguration is a combination of items of equipment, as specified by the manufacturer, that operate together as \na device to achieve an intended purpose. The combination of items may be modified, adjusted or customized to \nmeet specific needs. \nConfigurations include inter alia: \n—  gantries, tubes, tables, consoles and other items of equipment that can be configured/combined to deliver an \nintended function in computed tomography. \n—  ventilators, breathing circuits, vaporizers combined to deliver an intended function in anaesthesia. \nUDI-DI \nThe UDI-DI is a unique numeric or alphanumeric code specific to a model of device and that is also used as the \n'access key' to information stored in a UDI database. \nHuman Readable Interpretation ('HRI') \nHRI is a legible interpretation of the data characters encoded in the UDI carrier. \nPackaging levels \nPackaging levels means the various levels of device packaging that contain a defined quantity of devices, such as \na carton or case. \nUDI-PI \nThe UDI-PI is a numeric or alphanumeric code that identifies the unit of device production. \nThe different types of UDI-PIs include serial number, lot number, software identification and manufacturing or \nexpiry date or both types of date. \nUsefull Information to Consider:\nRadio Frequency Identification RFID \nRFID is a technology that uses communication through the use of radio waves to exchange data between \na reader and an electronic tag attached to an object, for the purpose of identification. \nShipping containers \nA shipping container is a container in relation to which traceability is controlled by a process specific to logistics \nsystems. \nUnique Device Identifier ('UDI') \nThe UDI is a series of numeric or alphanumeric characters that is created through a globally accepted device \nidentification and coding standard. It allows the unambiguous identification of a specific device on the market. \nThe UDI is comprised of the UDI-DI and the UDI-PI. \nThe word 'Unique' does not imply serialisation of individual production units. \nUDI carrier \nThe UDI carrier is the means of conveying the UDI by using AIDC and, if applicable, its HRI. \nUDI carriers include, inter alia, ID/linear bar code, 2D/Matrix bar code, RFID. "]
get_split_levels(w[0])

['simple']

In [390]:
def create_chunks(text, metadata):
    return {'page_content': text, 'metadata': metadata}

def split_document(text, metadata, levels, prefix = ""):
    if not levels or token_length(text) <= 512:
        if prefix:
            final_text = prefix + "\n" + text
        else:
            final_text = text
        new_metadata = metadata.copy()
        new_metadata['token_length'] = token_length(final_text)
        return [create_chunks(final_text, new_metadata)]

    current_level = levels[0]
    remaining_levels = levels[1:]
    pieces = get_text_piece(patterns[current_level],text)
    chunks = []
    print(f"="*200 + "\nFrom Pieces: \n",repr(pieces)+ "\n", "="*200)
    main_header = ""
    for piece in pieces:
        print("="*200 +"\nPiece: \n", piece+ "\n", "="*200)
        print("token lenght of current piece:",token_length(piece),"="*200)
        if any(re.match(pattern, piece) for pattern in marker_patterns.values()):
            continue
        print("="*200 +"\nThe Current Level is: \n",current_level + "\n", "="*200)
        match = re.match(patterns[current_level], piece)
        print("="*200 +"\nThe match for prefix: \n", match.group() if match else "")
        if match:
            header = match.group()
            new_prefix = prefix + "-" + header if prefix else "Chunk_Context: " + header
            piece = piece.replace(header,"")

        else:
            new_prefix = ""
        
    
        chunks.extend(split_document(piece, metadata, remaining_levels, prefix=new_prefix))
    return chunks

In [391]:
info = docs[147].get('page_content', '')
meta = docs[147].get("metadata")
level = get_split_levels(info)
split_document(info, meta, level, prefix = "")


From Pieces: 
 ['ANNEX VI \nINFORMATION TO BE SUBMITTED UPON THE REGISTRATION OF DEVICES AND ECONOMIC \nOPERATORS IN ACCORDANCE WITH ARTICLES 29(4) AND 31, CORE DATA ELEMENTS TO BE PROVIDED \nTO THE UDI DATABASE TOGETHER WITH THE UDI-DI IN ACCORDANCE WITH ARTICLES 28 AND 29, \nAND THE UDI SYSTEM ', '\nPART A \nINFORMATION TO BE SUBMITTED UPON THE REGISTRATION OF DEVICES AND ECONOMIC \nOPERATORS IN ACCORDANCE WITH ARTICLES 29(4) AND 31 \nManufacturers or, when applicable, authorised representatives, and, when applicable, importers shall submit the \ninformation referred to in Section 1 and shall ensure that the information on their devices referred to in Section 2 is \ncomplete, correct and updated by the relevant party. \n1.  \nInformation relating to the economic operator \n1.1.  \ntype of economic operator(manufacturer, authorised representative, or importer), \n1.2.  \nname, address and contact details of the economic operator, \n1.3.  \nwhere submission of information is carried ou

[{'page_content': '\nOPERATORS IN ACCORDANCE WITH ARTICLES 29(4) AND 31 \nManufacturers or, when applicable, authorised representatives, and, when applicable, importers shall submit the \ninformation referred to in Section 1 and shall ensure that the information on their devices referred to in Section 2 is \ncomplete, correct and updated by the relevant party. ',
  'metadata': {'document_name': 'eu_mdr_2017-745.pdf',
   'document_type': 'Regulations',
   'chapter': '',
   'chapter_title': '',
   'article': '',
   'article_title': '',
   'annex': 'ANNEX VI ',
   'annex_title': 'INFORMATION TO BE SUBMITTED UPON THE REGISTRATION OF DEVICES AND ECONOMIC ',
   'section': '',
   'section_title': '',
   'page_number': 115,
   'cross_references': '',
   'token_length': 63,
   'total_chunk': 1,
   'chunk_id': 1}},
 {'page_content': 'Chunk_Context: \nPART A \nINFORMATION TO BE SUBMITTED UPON THE REGISTRATION OF DEVICES AND ECONOMIC -\n1.  \nInformation relating to the economic operator \n\n1.1. 

In [ ]:
pieces = get_text_piece(patterns['simple'], a[-1])
print(len(pieces))
for i, p in enumerate(pieces):
    print(f"Piece {i}: {repr(p[:50])}, tokens: {token_length(p)}")

7
Piece 0: '\nPART C \nTHE UDI SYSTEM ', tokens: 8
Piece 1: '\n1.  \nDefinitions \nAutomatic identification and da', tokens: 707
Piece 2: '\n2.  \nGeneral requirements \n2.1.  \nThe affixing of', tokens: 105
Piece 3: '\n3.  \nThe UDI \n3.1.  \nA UDI shall be assigned to t', tokens: 516
Piece 4: '\n4.  \nUDI carrier \n4.1.  \nThe UDI carrier (AIDC an', tokens: 757
Piece 5: '\n5.  \nGeneral principles of the UDI database \n5.1.', tokens: 337
Piece 6: '\n6.  \nRules for specific device types \n6.1.  \nImpl', tokens: 1221


In [72]:
idx = a[-1].find('1.')
print(repr(a[-1][idx-2:idx+20]))

' \n1.  \nDefinitions \nAu'


In [74]:
pieces = get_text_piece(patterns['simple'], a[-1])
print(len(pieces))
for i, p in enumerate(pieces):
    print(f"Piece {i}: {repr(p[:50])}, tokens: {token_length(p)}")

7
Piece 0: '\nPART C \nTHE UDI SYSTEM ', tokens: 8
Piece 1: '\n1.  \nDefinitions \nAutomatic identification and da', tokens: 707
Piece 2: '\n2.  \nGeneral requirements \n2.1.  \nThe affixing of', tokens: 105
Piece 3: '\n3.  \nThe UDI \n3.1.  \nA UDI shall be assigned to t', tokens: 516
Piece 4: '\n4.  \nUDI carrier \n4.1.  \nThe UDI carrier (AIDC an', tokens: 757
Piece 5: '\n5.  \nGeneral principles of the UDI database \n5.1.', tokens: 337
Piece 6: '\n6.  \nRules for specific device types \n6.1.  \nImpl', tokens: 1221


In [75]:
piece_1 = pieces[1]
print(token_length(piece_1))
sub_chunks = split_document(piece_1, meta, ['decimal', 'triple'])
print(len(sub_chunks))
print([c['metadata']['token_length'] for c in sub_chunks])

707
0
[]


In [76]:
print(patterns['decimal'])
print(re.findall(patterns['decimal'], piece_1, re.M))


\d+\.\d+\.\s*

[]
